# Training the LLM Reranker

Run the following cell to install the package and dependencies if running in Google Colab.
If running locally, ensure you have installed the package via `pip install -e .`

*See also [LLM Ranker docs](docs/LLM_ranker.md) for tested environments*

In [ ]:
!git clone https://github.com/zheliu17/nanoRecSys.git
%pip install -q -e ./nanoRecSys[llm]

import psutil  # noqa: F401

# In fact, we don't need psutil; force-reinstalling it triggers a Colab restart.
%pip install --force-reinstall psutil=={psutil.__version__}
print("Installation complete. Please restart runtime...")

### 1. Dataset Preparation, Generate Embeddings, Negative Mining

Make sure `item_tower.pth` and `user_tower.pth` are in the `artifacts` directory (`nanoRecSys.config.settings.artifacts_dir`).

*See [Sequential Transformer Training](./sequential_transformer.ipynb) or [Metaflow Pipeline](../pipeline.py) for details on how to train/download the checkpoints.*

In [ ]:
# Unsloth should be imported as the first package

# import nanoRecSys.data.build_dataset
# import nanoRecSys.data.splits

# nanoRecSys.data.build_dataset.process_data()
# nanoRecSys.data.splits.create_user_time_split()
# nanoRecSys.data.build_dataset.prebuild_sequential_files()

# !wget -nv -P nanoRecSys/artifacts/ https://huggingface.co/zheliu97/nanoRecSys/resolve/main/{item,user}_tower.pth

import nanoRecSys.indexing.build_embeddings
from nanoRecSys.training.mine_negatives_sasrec import run_pipeline

nanoRecSys.indexing.build_embeddings.build_item_embeddings(batch_size=128)

# This will pair each interaction (20M) in the training set with one hard negative.
# We don't need 20M interactions for training the LLM ranker.
# 0.1 sampling_ratio is absolutely sufficient.
run_pipeline(
    batch_size=128,
    top_k=100,
    skip_top=0,
    sampling_ratio=0.1,
    suffix="llm_ranker",
    save_history=True,
    num_random_negatives=2,
    save_embeddings=False,  # LLM ranker doesn't use user embeddings
)

### 2. Two-Stage LLM Ranker Training

**Stage 1**: Embedding alignment. Projection training only, no LoRA.


**Stage 2**: Co-adaptation. Train projection and LLM together with LoRA.


About 10k steps in total. Takes about 3 hours on one A100 GPU.


Two-stage training is recommended when total training steps are small (e.g., 10k). If you have access to better compute, consider training for more steps (e.g., 50k), larger batch size (e.g., 64), and direct fine-tuning (skipping stage 1) for better performance.

*Checkpoints are available at <https://huggingface.co/zheliu97/nanoRecSys/tree/main>*

In [ ]:
from nanoRecSys.config import settings
from nanoRecSys.training.train_llm_ranker import LLMTrainingConfig, main as train_main

# You may download the weights to skip the whole training process
# model weights (adapter_model.safetensors) can be large
# Because we saved the modified embedding table (to add a special token)
# The provided weights don't work with newer versions
# Downgrading is required to load the checkpoints (see docs/LLM_ranker.md for details)

# !pip install transformers==4.57.2 unsloth==2025.11.1 unsloth_zoo==2025.11.2
# from huggingface_hub import hf_hub_download
# repo_id = 'zheliu97/nanoRecSys'
# hf_hub_download(repo_id=repo_id, filename='projection.pth', local_dir=settings.llm_output_dir)
# hf_hub_download(repo_id=repo_id, filename='adapter_model.safetensors', local_dir=settings.llm_output_dir)

config = LLMTrainingConfig(
    n_samples=int(10_000),  # about 1200 steps; enough for loss stabilization
    batch_size=32,
    learning_rate=1e-4,
    resume_from_checkpoint=False,
    warmup_steps=100,
    use_lora=False,
    report_to="none",  # i.e. 'wandb'
)

train_main(config)

# Set your projection weights path (*.pth); default is settings.llm_output_dir / "projection.pth"
projection_path = str(settings.llm_output_dir / "projection.pth")

config = LLMTrainingConfig(
    n_samples=int(80_000),  # about 9000 steps
    batch_size=32,
    learning_rate=5e-5,
    projection_path=projection_path,
    warmup_steps=1000,
    use_lora=True,
    report_to="none",  # i.e. 'wandb'
)

train_main(config)

### 3. Evaluation

In [ ]:
from nanoRecSys.eval.eval_llm_rankers import evaluate
from nanoRecSys.utils.utils import format_results_to_dataframe

results = evaluate(
    method="local",  # Evaluate the LLM ranker
    num_users=500,
    local_batch_size=16,
    use_lora=True,
    adapter_path=str(settings.llm_output_dir),
)

format_results_to_dataframe(results)